# MCI-GRU: Train -> Test metrics -> Backtest -> Download

Workflow:

1. **Setup** - detect Colab, clone repo, install deps, resolve root
2. **GPU preflight** - fail early if the Colab runtime is not on CUDA
3. **FRED key** - Colab Secrets or manual paste (required for regime features)
4. **Data** - market CSV via Drive mount, file upload, or local path
5. **Config** - experiment hyperparameters plus Phase 4 evaluation settings
6. **Train** - `run_experiment.py` (Hydra); outputs under `results/<name>/<timestamp>/`
7. **Inspect** - `training_summary.json`, `evaluation_summary.json`, `feature_reference.json`, `run_metadata.json`, `averaged_predictions/`, and helpers used by the offline bundle
8. **Backtest** - `tests/backtest_sp500.py --auto_save`
9. **Package** - materialize `offline_bundle/` inside the run (Hydra snapshot, MLflow export, optional FS mirror, manifest), zip the run tree, optionally zip the whole `mlruns/` store, then `files.download`

> **Colab runtime:** Switch to GPU before running - *Runtime -> Change runtime type -> T4 GPU*.

**Phase 4 outputs:** Training now writes `evaluation_summary.json` with IC/top-k/Newey-West Sharpe/bootstrap CI metrics and `feature_reference.json` with train-window drift references. Both are included in the final run archive.

**Label horizon:** `model.label_t` = what the model predicts (e.g. 21-day rank). `--label_t` in the backtest = return window for the simulation. They need not match.

**Recovery:** After download, unzip the run folder and read `offline_bundle/README.txt` for what was captured and how to open MLflow offline.


## 1. Environment, clone, and install


In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

# â”€â”€ Colab detection â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
try:
    from google.colab import files as colab_files  # type: ignore
    IN_COLAB = True
except ImportError:
    colab_files = None
    IN_COLAB = False

COLAB_REPO_DIR = Path("/content/MCI-GRU")
MANUAL_REPO_ROOT = None  # Local override: e.g. Path(r"C:\Users\you\MCI-GRU")

# â”€â”€ Clone / pull (Colab only) â€” runs BEFORE resolve_repo_root â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
REPO_URL = "https://github.com/magilliam27/MCI-GRU.git"
BRANCH   = "main"  # change to your branch if needed

if IN_COLAB:
    if not COLAB_REPO_DIR.exists():
        print(f"Cloning {REPO_URL} @ {BRANCH} â€¦")
        subprocess.run(
            ["git", "clone", "-b", BRANCH, REPO_URL, str(COLAB_REPO_DIR)],
            check=True,
        )
    else:
        print("Repo already cloned â€” pulling latest â€¦")
        subprocess.run(["git", "-C", str(COLAB_REPO_DIR), "fetch", "origin"], check=False)
        subprocess.run(["git", "-C", str(COLAB_REPO_DIR), "checkout", BRANCH], check=False)
        subprocess.run(["git", "-C", str(COLAB_REPO_DIR), "pull", "origin", BRANCH], check=False)
    print("Installing dependencies â€¦")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(COLAB_REPO_DIR / "requirements.txt")],
        check=True,
    )
else:
    print("Local environment â€” skipped clone/install.")

# â”€â”€ Resolve repo root â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def resolve_repo_root() -> Path:
    if MANUAL_REPO_ROOT is not None:
        return Path(MANUAL_REPO_ROOT).resolve()
    cwd = Path.cwd().resolve()
    for p in (cwd, cwd.parent, cwd.parent.parent):
        if (p / "run_experiment.py").is_file():
            return p
    if IN_COLAB and COLAB_REPO_DIR.is_dir():
        return COLAB_REPO_DIR.resolve()
    raise FileNotFoundError(
        "Cannot find run_experiment.py. Set MANUAL_REPO_ROOT or clone the repo on Colab."
    )

REPO_ROOT = resolve_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("REPO_ROOT =", REPO_ROOT)
print("IN_COLAB  =", IN_COLAB)


## 1b. GPU preflight

This notebook is intended for Colab GPU runtimes. Run this before starting the long training cell so a CPU runtime fails quickly.


In [ ]:
REQUIRE_GPU = True  # Set False only for a deliberate local/CPU dry run.

try:
    import torch
except ImportError as exc:
    raise RuntimeError("PyTorch is not installed; rerun the setup/install cell first.") from exc

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    device_idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_idx)
    print("CUDA version:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(device_idx))
    print(f"GPU memory: {props.total_memory / (1024 ** 3):.1f} GiB")
else:
    msg = (
        "No CUDA GPU is visible. In Colab, choose Runtime -> Change runtime type -> GPU, "
        "then restart and run from the top."
    )
    if REQUIRE_GPU:
        raise RuntimeError(msg)
    print("WARNING:", msg)


## 2. FRED API key (required for regime features)


In [ ]:
# Option A (recommended for Colab): add a Secret named FRED_API_KEY
#   Secrets panel â‰¡ â†’ ðŸ”‘ â†’ "Add new secret" â†’ name: FRED_API_KEY
# Option B: paste the key directly below.
MY_FRED_KEY = ""  # paste key here if not using Colab Secrets

if not os.environ.get("FRED_API_KEY"):
    if IN_COLAB:
        try:
            from google.colab import userdata  # type: ignore
            _secret = userdata.get("FRED_API_KEY")
            if _secret:
                os.environ["FRED_API_KEY"] = _secret
                print("FRED_API_KEY loaded from Colab Secrets.")
        except Exception:
            pass

    if not os.environ.get("FRED_API_KEY") and MY_FRED_KEY.strip():
        os.environ["FRED_API_KEY"] = MY_FRED_KEY.strip()
        print("FRED_API_KEY set from MY_FRED_KEY.")

if not os.environ.get("FRED_API_KEY"):
    raise ValueError(
        "FRED_API_KEY is not set. "
        "Add it as a Colab Secret (key: FRED_API_KEY) or paste it into MY_FRED_KEY above."
    )
print(f"FRED_API_KEY is set (length {len(os.environ['FRED_API_KEY'])}).")


## 2b. MLflow tracking

MLflow is already installed via `requirements.txt` (no extra pip install needed). The repo defaults to **`tracking.enabled=true`**; in this notebook, **`ENABLE_MLFLOW=False`** adds `tracking.enabled=false` so training does not log to MLflow.

| `MLFLOW_TRACKING_URI` | Where runs are stored |
|---|---|
| `"mlruns"` *(default)* | Local store under repo root; run zip includes `offline_bundle/` mirror; separate zip bundles all of `mlruns/` |
| `"https://dagshub.com/<user>/<repo>.mlflow"` | DagsHub remote — persists after the runtime ends |
| `"http://<host>:5000"` | Self-hosted MLflow server |

Cell 8 materializes `offline_bundle/` inside the run (Hydra snapshot, MLflow JSON export, optional run FS mirror), zips the run, and for local file stores also zips `mlruns/`. Open `offline_bundle/README.txt` in the run zip after download.

In [ ]:
# ── MLflow tracking configuration ─────────────────────────────────────────
ENABLE_MLFLOW = True          # set False to skip all MLflow tracking

# Tracking store — choose one:
#   "mlruns"                                   local, bundled in zip (default)
#   "https://dagshub.com/<user>/<repo>.mlflow" remote, persists after runtime
#   "http://<host>:5000"                       self-hosted server
MLFLOW_TRACKING_URI = "mlruns"

# Optional: override the MLflow experiment name (defaults to EXPERIMENT_NAME)
MLFLOW_EXPERIMENT_NAME = None   # e.g. "colab-runs"

# ── Resolve absolute path / URI for subprocess calls ─────────────────────
if ENABLE_MLFLOW:
    if "://" in MLFLOW_TRACKING_URI:
        _mlflow_uri_abs = MLFLOW_TRACKING_URI          # already a remote URI
    else:
        _mlruns_dir = REPO_ROOT / MLFLOW_TRACKING_URI
        _mlruns_dir.mkdir(parents=True, exist_ok=True)
        _mlflow_uri_abs = str(_mlruns_dir)             # absolute local path
    print(f"MLflow enabled      : {ENABLE_MLFLOW}")
    print(f"MLflow tracking URI : {_mlflow_uri_abs}")
else:
    _mlflow_uri_abs = None
    print("MLflow disabled — set ENABLE_MLFLOW=True to activate.")


def _mlflow_is_remote_uri(uri: str | None) -> bool:
    """True for http(s) tracking servers. Local paths may contain ':' (Windows drive)."""
    if not uri:
        return False
    v = str(uri).strip().lower()
    return v.startswith("http://") or v.startswith("https://")


## 3. Market data CSV

Three ways to provide the market universe CSV:

| Option | How |
|--------|-----|
| **Drive mount** | Set `USE_GOOGLE_DRIVE = True` and `DRIVE_CSV_RELPATH` (recommended for large files) |
| **File upload** | Leave defaults â€” Colab prompts for upload on first run |
| **Local path** | Set `LOCAL_MARKET_CSV` to an absolute path (when running locally) |

Regime inputs are **not** read from a file: `features=with_momentum` with `include_global_regime=true` leaves `regime_inputs_csv` null, so `load_regime_inputs` pulls macro series from the FRED API.


In [ ]:
# â”€â”€ Configure one of the three options â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
USE_GOOGLE_DRIVE  = False   # True â†’ mount Drive; False â†’ upload or local path
DRIVE_CSV_RELPATH = "sp500_2019_universe_data_through_2026.csv"  # path inside MyDrive
LOCAL_MARKET_CSV  = None    # e.g. Path(r"C:\data\sp500.csv")  â€” local runs only

# â”€â”€ Resolution (no need to edit below) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
DEFAULT_CSV_NAME = "sp500_2019_universe_data_through_2026.csv"
CSV_DIR = REPO_ROOT / "data" / "raw" / "market"
CSV_DIR.mkdir(parents=True, exist_ok=True)

if LOCAL_MARKET_CSV is not None:
    MARKET_CSV = Path(LOCAL_MARKET_CSV).resolve()
elif IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive", force_remount=False)
    MARKET_CSV = Path("/content/drive/MyDrive") / DRIVE_CSV_RELPATH
elif IN_COLAB:
    MARKET_CSV = CSV_DIR / DEFAULT_CSV_NAME
    if not MARKET_CSV.is_file():
        print("Upload your market universe CSV (saved as MARKET_CSV) â€¦")
        up = colab_files.upload()
        if not up:
            raise RuntimeError("Upload cancelled or empty.")
        MARKET_CSV.write_bytes(next(iter(up.values())))
        print(f"Saved to {MARKET_CSV}")
else:
    MARKET_CSV = CSV_DIR / DEFAULT_CSV_NAME

if not MARKET_CSV.is_file():
    raise FileNotFoundError(f"Market CSV not found: {MARKET_CSV}")
print("MARKET_CSV OK:", MARKET_CSV)


## 4. Experiment configuration

Correlation graph: set **`GRAPH_PROFILE`** in the next cell to **`dynamic`** (uses `GRAPH_*` knobs) or **`legacy_baseline`** (static graph, no `GraphSchedule`, threshold edges, scalar `edge_weight` — ignores `GRAPH_*` for Hydra). **Defaults** use the dynamic top-20 + 4-D edge preset. Rename **`EXPERIMENT_NAME`** between A/B runs so `results/` stays organized.

**Label-time embargo:** `ExperimentConfig` requires calendar gaps **strictly greater than `LABEL_T`** between `train_end`→`val_start` and `val_end`→`test_start`. If you raise **`LABEL_T`** or switch `data=…` presets, the next cell may auto-append `data.val_start` / `data.test_start` overrides after reading `configs/data/<preset>.yaml`.


In [ ]:
# Mirrors Seed_test BASE: with_momentum + dynamic momentum + global regime.
EXPERIMENT_NAME = "notebook_train_backtest"
SEED        = 7
NUM_MODELS  = 10
HIS_T       = 60
LABEL_T     = 21

# --- Correlation graph (Hydra graph.*) ------------------------------------
# GRAPH_PROFILE:
#   "dynamic"          — use GRAPH_* below (scheduled graph when months>0, top-K, multi-feature).
#   "legacy_baseline"  — A/B control: static graph, threshold edges, scalar weights (GRAPH_* ignored).
GRAPH_PROFILE = "dynamic"  # "dynamic" | "legacy_baseline"

# Used only when GRAPH_PROFILE == "dynamic" (defaults = recommended dynamic top-K run).
GRAPH_UPDATE_FREQUENCY_MONTHS = 6       # 0 = static; >0 = GraphSchedule every N months
GRAPH_TOP_K = 20                        # 0 = legacy corr > judge_value; >0 = per-node top-K
GRAPH_TOP_K_METRIC = "corr"           # "corr" (positive-only K) or "abs_corr" (|corr|, signed in tensor)
GRAPH_USE_MULTI_FEATURE_EDGES = True   # True => (E,4) edge attr; run_experiment resolves final edge_feature_dim

# --- Phase 4 evaluation (Hydra evaluation.*) ---------------------------
EVALUATION_TOP_K_VALUES = [10, 20, 50, 100]
EVALUATION_BOOTSTRAP_ENABLED = True
EVALUATION_BOOTSTRAP_RESAMPLES = 1000
EVALUATION_CI_LEVEL = 0.95
EVALUATION_SHARPE_METHOD = "newey_west"

if GRAPH_PROFILE == "legacy_baseline":
    _GRAPH_MONTHS = 0
    _GRAPH_TOP_K = 0
    _GRAPH_TOP_K_METRIC = "corr"
    _GRAPH_MULTI = False
elif GRAPH_PROFILE == "dynamic":
    _GRAPH_MONTHS = GRAPH_UPDATE_FREQUENCY_MONTHS
    _GRAPH_TOP_K = GRAPH_TOP_K
    _GRAPH_TOP_K_METRIC = GRAPH_TOP_K_METRIC
    _GRAPH_MULTI = GRAPH_USE_MULTI_FEATURE_EDGES
else:
    raise ValueError(
        f'GRAPH_PROFILE must be "dynamic" or "legacy_baseline", got {GRAPH_PROFILE!r}'
    )

print(
    f"Graph profile: {GRAPH_PROFILE}  ->  Hydra graph: "
    f"months={_GRAPH_MONTHS}, top_k={_GRAPH_TOP_K}, "
    f"top_k_metric={_GRAPH_TOP_K_METRIC!r}, use_multi_feature_edges={_GRAPH_MULTI}"
)
print(
    "Phase 4 evaluation -> "
    f"top_k={EVALUATION_TOP_K_VALUES}, bootstrap={EVALUATION_BOOTSTRAP_ENABLED}, "
    f"resamples={EVALUATION_BOOTSTRAP_RESAMPLES}, sharpe={EVALUATION_SHARPE_METHOD}"
)

rel_market = MARKET_CSV.relative_to(REPO_ROOT).as_posix()

HYDRA_OVERRIDES = [
    f"experiment_name={EXPERIMENT_NAME}",
    f"seed={SEED}",
    # Use 'data=temporal_2019' (not '+data=') to override the default data config.
    "data=temporal_2019",
    "data.source=csv",
    f"data.filename={rel_market}",
    "features=with_momentum",
    "features.include_weekly_momentum=false",
    "features.momentum_encoding=binary",
    "features.momentum_blend_mode=dynamic",
    "features.momentum_dynamic_correction_fast_weight=0.15",
    "features.momentum_dynamic_rebound_fast_weight=0.7",
    "features.momentum_dynamic_lookback_periods=0",
    "features.momentum_dynamic_min_history=252",
    "features.momentum_dynamic_min_state_observations=3",
    "features.include_global_regime=true",
    "features.regime_enforce_lag_days=0",
    f"model.his_t={HIS_T}",
    f"model.label_t={LABEL_T}",
    "model.use_multi_scale=false",
    "model.use_self_attention=false",
    "model.activation=elu",
    "model.latent_init_scale=0.02",
    "training.label_type=rank",
    f"training.num_models={NUM_MODELS}",
    "training.num_epochs=100",
    "training.early_stopping_patience=15",
    f"graph.update_frequency_months={_GRAPH_MONTHS}",
    f"graph.top_k={_GRAPH_TOP_K}",
    f"graph.top_k_metric={_GRAPH_TOP_K_METRIC}",
    f"graph.use_multi_feature_edges={str(_GRAPH_MULTI).lower()}",
    f"evaluation.top_k_values=[{','.join(str(k) for k in EVALUATION_TOP_K_VALUES)}]",
    f"evaluation.bootstrap_enabled={str(EVALUATION_BOOTSTRAP_ENABLED).lower()}",
    f"evaluation.bootstrap_resamples={EVALUATION_BOOTSTRAP_RESAMPLES}",
    f"evaluation.ci_level={EVALUATION_CI_LEVEL}",
    f"evaluation.sharpe_method={EVALUATION_SHARPE_METHOD}",
]

# ── Embargo: gaps (val_start - train_end) and (test_start - val_end) must be > LABEL_T ──
import yaml as _yaml_embargo
from datetime import date as _date_embargo, timedelta as _timedelta_embargo

_data_preset = None
for _o in HYDRA_OVERRIDES:
    if _o.startswith("data=") and _o.count("=") == 1:
        _data_preset = _o.split("=", 1)[1]
        break
if _data_preset:
    _yp = REPO_ROOT / "configs" / "data" / f"{_data_preset}.yaml"
    if _yp.is_file():
        _dd = _yaml_embargo.safe_load(_yp.read_text(encoding="utf-8"))
        _te = _date_embargo.fromisoformat(str(_dd["train_end"]))
        _ve = _date_embargo.fromisoformat(str(_dd["val_end"]))
        _vs = _date_embargo.fromisoformat(str(_dd["val_start"]))
        _ts = _date_embargo.fromisoformat(str(_dd["test_start"]))
        if (_vs - _te).days <= LABEL_T:
            _vs = _te + _timedelta_embargo(days=LABEL_T + 1)
            HYDRA_OVERRIDES.append(f"data.val_start={_vs.isoformat()}")
            print(f"Embargo: data.val_start -> {_vs.isoformat()} (gap > {LABEL_T} after train_end)")
        if (_ts - _ve).days <= LABEL_T:
            _ts = _ve + _timedelta_embargo(days=LABEL_T + 1)
            HYDRA_OVERRIDES.append(f"data.test_start={_ts.isoformat()}")
            print(f"Embargo: data.test_start -> {_ts.isoformat()} (gap > {LABEL_T} after val_end)")

# ── MLflow tracking overrides ─────────────────────────────────────────────
if ENABLE_MLFLOW and _mlflow_uri_abs:
    _mlflow_exp = MLFLOW_EXPERIMENT_NAME or EXPERIMENT_NAME
    HYDRA_OVERRIDES += [
        "tracking.enabled=true",
        f"tracking.tracking_uri={_mlflow_uri_abs}",
        f"tracking.experiment_name={_mlflow_exp}",
        "tracking.log_artifacts=true",
        "tracking.log_predictions=true",
    ]
else:
    HYDRA_OVERRIDES.append("tracking.enabled=false")

print(f"Hydra overrides ({len(HYDRA_OVERRIDES)} args):")
for o in HYDRA_OVERRIDES:
    print("  ", o)

## 5. Train


In [ ]:
RUN_TRAIN = True

_TAIL_LINES = 120          # rolling tail kept in memory for focused error reporting
_STREAM_PREFIX = "  | "    # marks lines that came from the training subprocess


def _format_elapsed(seconds: float) -> str:
    """Render a wall-clock duration as e.g. '1h 03m 12s' / '4m 07s' / '9s'."""
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h:d}h {m:02d}m {s:02d}s"
    if m:
        return f"{m:d}m {s:02d}s"
    return f"{s:d}s"


def _extract_traceback(lines):
    """Return the last Python traceback in `lines`, or None if absent.

    Anchored on the most recent 'Traceback (most recent call last):' so that
    chained / re-raised tracebacks still surface the final, actionable one.
    """
    start = None
    for i, line in enumerate(lines):
        if line.startswith("Traceback (most recent call last):"):
            start = i
    if start is None:
        return None
    return "\n".join(lines[start:])


def run_training():
    # '-u' forces the child Python to flush stdout per line; without it Hydra/
    # logging output is block-buffered when stdout is a pipe, so the notebook
    # would show nothing until the subprocess exits.
    cmd = [sys.executable, "-u", str(REPO_ROOT / "run_experiment.py"), *HYDRA_OVERRIDES]
    started_at = datetime.now()

    print("=" * 80)
    print(f"Training started at {started_at.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"REPO_ROOT      : {REPO_ROOT}")
    print(f"Python         : {sys.executable}")
    print(f"Script         : run_experiment.py")
    print(f"Hydra overrides: {len(HYDRA_OVERRIDES)}")
    for o in HYDRA_OVERRIDES[:8]:
        print(f"   - {o}")
    if len(HYDRA_OVERRIDES) > 8:
        print(f"   ... (+{len(HYDRA_OVERRIDES) - 8} more)")
    print("-" * 80)
    print("Streaming subprocess output (prefix '|') ...")
    print("-" * 80)

    tail = []                # rolling buffer of recent stdout lines
    output_dir_seen = None   # captured from run_experiment.py's own log line

    proc = subprocess.Popen(
        cmd,
        cwd=str(REPO_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,           # line-buffered on the parent side
    )
    try:
        assert proc.stdout is not None
        for raw in proc.stdout:
            line = raw.rstrip("\n")
            print(_STREAM_PREFIX + line, flush=True)
            tail.append(line)
            if len(tail) > _TAIL_LINES:
                del tail[0 : len(tail) - _TAIL_LINES]
            if output_dir_seen is None and "Output directory:" in line:
                output_dir_seen = line.split("Output directory:", 1)[1].strip()
        returncode = proc.wait()
    except KeyboardInterrupt:
        # Don't leave an orphan training process behind when the user cancels.
        print("\nKeyboardInterrupt received - terminating training subprocess ...")
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait()
        raise

    elapsed = (datetime.now() - started_at).total_seconds()
    print("-" * 80)
    print(f"Subprocess exited with code {returncode} after {_format_elapsed(elapsed)}.")

    if returncode != 0:
        tb = _extract_traceback(tail)
        print("=" * 80)
        print("TRAINING FAILED")
        print("=" * 80)
        print(f"Exit code      : {returncode}")
        print(f"Elapsed        : {_format_elapsed(elapsed)}")
        print(f"Command        : {' '.join(cmd[:3])} ... (+{len(HYDRA_OVERRIDES)} overrides)")
        if tb:
            print("\nLast Python traceback from subprocess:")
            print("-" * 80)
            print(tb)
            print("-" * 80)
        else:
            # No Python traceback => likely OOM-kill, segfault, or non-Python failure.
            print(f"\nLast {len(tail)} line(s) of subprocess output (no Python traceback found):")
            print("-" * 80)
            print("\n".join(tail))
            print("-" * 80)
        raise RuntimeError(
            f"Training failed (exit code {returncode}) after {_format_elapsed(elapsed)}. "
            f"See the traceback / tail above; the full streamed log is in this cell's output."
        )

    print("=" * 80)
    print("TRAINING SUCCEEDED")
    print("=" * 80)
    print(f"Elapsed        : {_format_elapsed(elapsed)}")
    if output_dir_seen:
        print(f"Output dir     : {output_dir_seen}")


if RUN_TRAIN:
    run_training()
else:
    print("Skipped training (RUN_TRAIN=False).")


## 6. Inspect latest run and Phase 4 artifacts


In [ ]:
RESULTS_ROOT = REPO_ROOT / "results"


def is_hydra_timestamp_dir(name: str) -> bool:
    return bool(re.fullmatch(r"\d{8}_\d{6}", name))


def find_latest_run(results_root: Path, experiment_name: str) -> Path:
    base = results_root / experiment_name
    if not base.is_dir():
        raise FileNotFoundError(base)
    candidates = [p for p in base.iterdir() if p.is_dir() and is_hydra_timestamp_dir(p.name)]
    if not candidates:
        raise FileNotFoundError(f"No runs under {base}")
    return sorted(candidates)[-1]


LATEST_RUN = find_latest_run(RESULTS_ROOT, EXPERIMENT_NAME)
PRED_DIR   = LATEST_RUN / "averaged_predictions"

print("LATEST_RUN:", LATEST_RUN)
print("PRED_DIR:  ", PRED_DIR)

ts = LATEST_RUN / "training_summary.json"
if ts.is_file():
    with open(ts, encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2)[:6000])
else:
    print("No training_summary.json")

es = LATEST_RUN / "evaluation_summary.json"
if es.is_file():
    print("\nevaluation_summary.json:")
    with open(es, encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2)[:6000])
else:
    print("No evaluation_summary.json")

fr = LATEST_RUN / "feature_reference.json"
if fr.is_file():
    with open(fr, encoding="utf-8") as f:
        feature_ref = json.load(f)
    feature_count = len(feature_ref.get("features", {}))
    print(f"feature_reference.json OK: {feature_count} features")
else:
    print("No feature_reference.json")

rm = LATEST_RUN / "run_metadata.json"
if rm.is_file():
    with open(rm, encoding="utf-8") as f:
        meta = json.load(f)
    print("run_metadata keys:", sorted(meta.keys()))
    print("feature_reference_path:", meta.get("feature_reference_path"))


def capture_run_for_local_recovery(
    latest_run: Path,
    *,
    hydra_overrides: list[str],
    experiment_name: str,
    seed: int,
    mirror_mlflow_run_fs: bool = True,
) -> Path:
    """Write ``offline_bundle/`` under the Hydra run dir so one zip is self-contained.

    Includes: notebook Hydra overrides, optional git HEAD, NumPy/Torch versions,
    full ``.hydra/`` snapshot, MLflow params + latest metrics + tags as JSON,
    optional copy of the MLflow on-disk run tree (``file:`` stores only),
    file tree manifest, backtest subprocess log if present, and a short README.
    """
    from datetime import datetime, timezone
    from urllib.parse import urlparse

    latest_run = Path(latest_run).resolve()
    bundle = latest_run / "offline_bundle"
    if bundle.is_dir():
        shutil.rmtree(bundle)
    bundle.mkdir(parents=True)

    ctx = {
        "captured_at_utc": datetime.now(timezone.utc).isoformat(),
        "experiment_name": experiment_name,
        "seed": seed,
        "hydra_overrides": list(hydra_overrides),
        "repo_root": str(REPO_ROOT.resolve()),
        "latest_run": str(latest_run),
    }
    try:
        ctx["market_csv"] = str(MARKET_CSV.resolve())
    except Exception:
        ctx["market_csv"] = str(MARKET_CSV)
    try:
        import numpy as np

        ctx["numpy_version"] = np.__version__
    except Exception as exc:  # pragma: no cover - optional dep
        ctx["numpy_import_error"] = repr(exc)
    try:
        import torch

        ctx["torch_version"] = torch.__version__
        ctx["cuda_available"] = bool(torch.cuda.is_available())
    except Exception as exc:  # pragma: no cover
        ctx["torch_import_error"] = repr(exc)

    try:
        cp = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
            capture_output=True,
            text=True,
            timeout=10,
            check=False,
        )
        if cp.returncode == 0:
            ctx["git_commit"] = cp.stdout.strip()
        cp2 = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "status", "--porcelain"],
            capture_output=True,
            text=True,
            timeout=10,
            check=False,
        )
        if cp2.returncode == 0:
            ctx["git_porcelain"] = cp2.stdout.strip() or "(clean)"
    except Exception as exc:  # pragma: no cover
        ctx["git_error"] = repr(exc)

    (bundle / "notebook_context.json").write_text(
        json.dumps(ctx, indent=2, default=str), encoding="utf-8"
    )

    hydra_src = latest_run / ".hydra"
    if hydra_src.is_dir():
        shutil.copytree(hydra_src, bundle / "hydra_snapshot")

    mlflow_path = latest_run / "mlflow_run.json"
    if mlflow_path.is_file():
        meta = json.loads(mlflow_path.read_text(encoding="utf-8"))
        export: dict = {"mlflow_run.json": meta}
        try:
            from mlflow.tracking import MlflowClient

            uri = meta.get("tracking_uri")
            run_id = meta.get("run_id")
            if uri and run_id:
                client = MlflowClient(tracking_uri=uri)
                run = client.get_run(run_id)
                export["params"] = dict(run.data.params)
                export["metrics_latest"] = dict(run.data.metrics)
                export["tags"] = dict(run.data.tags)
        except Exception as exc:
            export["mlflow_client_error"] = repr(exc)

        (bundle / "mlflow_export.json").write_text(
            json.dumps(export, indent=2, default=str), encoding="utf-8"
        )

        if mirror_mlflow_run_fs:
            uri = meta.get("tracking_uri") or ""
            exp_id = meta.get("experiment_id")
            run_id = meta.get("run_id")
            if str(uri).lower().startswith("file:") and exp_id and run_id:
                path_part = urlparse(uri).path
                mlruns_root = Path(path_part)
                run_dir = mlruns_root / str(exp_id) / str(run_id)
                if run_dir.is_dir():
                    shutil.copytree(run_dir, bundle / "mlflow_run_fs_mirror")

    files: list[dict] = []
    skip_prefix = "offline_bundle"
    for p in sorted(latest_run.rglob("*")):
        if p.is_file():
            rel = p.relative_to(latest_run).as_posix()
            if rel == skip_prefix or rel.startswith(skip_prefix + "/"):
                continue
            try:
                files.append({"path": rel, "size_bytes": p.stat().st_size})
            except OSError:
                pass
    (bundle / "run_tree_manifest.json").write_text(
        json.dumps({"run_root": str(latest_run), "files": files}, indent=2),
        encoding="utf-8",
    )

    training_logs = sorted(latest_run.glob("training_*.log"))
    if training_logs:
        tlog = training_logs[-1]
        try:
            raw = tlog.read_bytes()
            if len(raw) > 2_000_000:
                (bundle / "training_log_tail.txt").write_bytes(raw[-2_000_000:])
                (bundle / "training_log_note.txt").write_text(
                    f"Truncated: last ~2MB of {tlog.name} (full file is on run root).",
                    encoding="utf-8",
                )
            else:
                shutil.copy2(tlog, bundle / tlog.name)
        except OSError as exc:
            (bundle / "training_log_note.txt").write_text(
                f"Could not copy training log: {exc}", encoding="utf-8"
            )

    bt_log = latest_run / "backtest" / "notebook_subprocess_stdout.log"
    if not bt_log.is_file():
        for alt in sorted(latest_run.glob("backtest*/notebook_subprocess_stdout.log")):
            bt_log = alt
            break
    if bt_log.is_file():
        (bundle / "backtest_subprocess_stdout.log").write_text(
            bt_log.read_text(encoding="utf-8", errors="replace"), encoding="utf-8"
        )

    readme = f"""Offline recovery bundle (written before zip/download)
=============================================================

Files:
- notebook_context.json   Notebook Hydra overrides, seed, git HEAD, library versions
- hydra_snapshot/         Copy of Hydra's .hydra/ (config.yaml, overrides.yaml, etc.)
- mlflow_export.json      mlflow_run.json + params + latest metrics + tags
- mlflow_run_fs_mirror/   Present only for file:// tracking URI — on-disk MLflow run tree
- run_tree_manifest.json  Inventory of files under the run root (excluding offline_bundle)
- backtest_subprocess_stdout.log  Copy of backtest cell stdout log (if backtest ran)
- training_*.log or training_log_tail.txt  Copy of Hydra training log (tail only if >2MB)
- evaluation_summary.json          Phase 4 evaluation metrics (IC, top-k, CIs, Newey-West Sharpe)
- feature_reference.json           Train-window feature drift reference bins/counts

Open MLflow locally (full store zip is also downloaded for local file:// tracking):
  mlflow ui --backend-store-uri ./mlruns

Or point the UI at a copied run mirror under mlflow_run_fs_mirror/ inside this bundle
if you only need one run offline.
"""
    (bundle / "README.txt").write_text(readme, encoding="utf-8")
    print("Offline bundle written to:", bundle)
    return bundle


## 7. Backtest


In [ ]:
BACKTEST_LABEL_T = 5
TEST_START       = "2025-01-01"
TEST_END         = "2025-12-31"
TOP_K            = 20
BACKTEST_SUFFIX  = ""

rel_market_bt = MARKET_CSV.relative_to(REPO_ROOT).as_posix()

bt_cmd = [
    sys.executable,
    str(REPO_ROOT / "tests" / "backtest_sp500.py"),
    "--predictions_dir", str(PRED_DIR),
    "--data_file",       rel_market_bt,
    "--test_start",      TEST_START,
    "--test_end",        TEST_END,
    "--top_k",           str(TOP_K),
    "--label_t",         str(BACKTEST_LABEL_T),
    "--enable_rank_drop_gate",
    "--min_rank_drop",   "30",
    "--transaction_costs",
    "--spread",          "5",
    "--auto_save",
    "--plot",
]
if BACKTEST_SUFFIX:
    bt_cmd.extend(["--backtest_suffix", BACKTEST_SUFFIX])

# ── MLflow: link backtest as a child run under the training parent ─────────
# backtest_sp500.py auto-links via mlflow_run.json if it exists in LATEST_RUN.
if ENABLE_MLFLOW and _mlflow_uri_abs:
    bt_cmd.extend(["--enable_mlflow", "--mlflow_tracking_uri", _mlflow_uri_abs])

_env = os.environ.copy()
_env["PYTHONPATH"] = str(REPO_ROOT) + os.pathsep + _env.get("PYTHONPATH", "")

print("Backtest:", " ".join(bt_cmd))
proc = subprocess.run(
    bt_cmd,
    cwd=str(REPO_ROOT),
    env=_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    raise RuntimeError(
        f"Backtest exit code {proc.returncode}. Full subprocess output printed above."
    )

bt_name = "backtest" + (BACKTEST_SUFFIX or "")
BACKTEST_DIR = LATEST_RUN / bt_name
BACKTEST_DIR.mkdir(parents=True, exist_ok=True)
(BACKTEST_DIR / "notebook_subprocess_stdout.log").write_text(
    proc.stdout or "", encoding="utf-8"
)
print("BACKTEST_DIR:", BACKTEST_DIR)
metrics_file = BACKTEST_DIR / "backtest_metrics.json"
if metrics_file.is_file():
    with open(metrics_file, encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2)[:4000])

In [ ]:
# ── MLflow run summary ────────────────────────────────────────────────────
from mci_gru.tracking.mlflow_manager import load_run_metadata

mlflow_meta = load_run_metadata(LATEST_RUN)
if mlflow_meta:
    print("MLflow run metadata:")
    print(json.dumps(mlflow_meta, indent=2))
    tracking_uri = mlflow_meta.get("tracking_uri", "")
    print()
    if not _mlflow_is_remote_uri(tracking_uri):
        # Local path or file:// — after download use mlruns zip or offline_bundle mirror
        print("To view runs locally after downloading the zip(s):")
        if str(tracking_uri).lower().startswith("file:"):
            from urllib.parse import urlparse

            p = urlparse(tracking_uri).path
            print(f"  mlflow ui --backend-store-uri {p}")
        else:
            print(f"  mlflow ui --backend-store-uri {tracking_uri}")
    else:
        print(f"Open {tracking_uri} in your browser to view runs.")
else:
    print("No MLflow metadata found (ENABLE_MLFLOW may be False, or training hasn't run yet).")

## 8. Offline bundle, zip, and download

Writes ``offline_bundle/`` into ``LATEST_RUN`` (Hydra snapshot, MLflow export, manifest, README), then zips the full run directory. For local file stores, a second zip contains the entire ``mlruns/`` tree.


In [ ]:
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Self-contained artifacts for local / drive recovery (Hydra, MLflow export, manifest).
capture_run_for_local_recovery(
    LATEST_RUN,
    hydra_overrides=HYDRA_OVERRIDES,
    experiment_name=EXPERIMENT_NAME,
    seed=SEED,
    mirror_mlflow_run_fs=True,
)

zip_base = REPO_ROOT / "results" / f"{EXPERIMENT_NAME}_bundle_{stamp}"
archive_path = shutil.make_archive(
    str(zip_base), "zip",
    root_dir=str(LATEST_RUN.parent),
    base_dir=LATEST_RUN.name,
)
print("Run archive:", archive_path)

if IN_COLAB and colab_files is not None:
    colab_files.download(archive_path)
else:
    print("Local: copy results from:", LATEST_RUN)

# ── Bundle the full local MLflow store (optional second download) ──────────
# _mlflow_is_remote_uri avoids treating Windows paths (C:\...) as remote.
if (
    ENABLE_MLFLOW
    and _mlflow_uri_abs is not None
    and not _mlflow_is_remote_uri(_mlflow_uri_abs)
):
    _mlruns_path = Path(_mlflow_uri_abs)
    if _mlruns_path.is_dir():
        mlruns_zip_base = REPO_ROOT / "results" / f"mlruns_{stamp}"
        mlruns_archive = shutil.make_archive(
            str(mlruns_zip_base), "zip",
            root_dir=str(_mlruns_path.parent),
            base_dir=_mlruns_path.name,
        )
        print("MLflow store archive (full):", mlruns_archive)
        print("After downloading, run: mlflow ui --backend-store-uri ./mlruns")
        if IN_COLAB and colab_files is not None:
            colab_files.download(mlruns_archive)